# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Umesh-MishraRB/FlyRank_ML_Intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
!pip install -q duckdb pandas numpy huggingface_hub requests pyarrow

In [7]:
import os
import time
import requests
import duckdb
import pandas as pd
import numpy as np
from getpass import getpass
from huggingface_hub import snapshot_download

# ---- Config ------------------------------------------------------------
FAST = True                      # True = June only (quick proof). False = Jan-Jun.
START_DATE = "2026-06-01" if FAST else "2026-01-01"
END_DATE = "2026-06-30"

RECENT_N = 7          # last N observations = "recent"
MIN_HISTORY = 3       # below this -> INSUFFICIENT_EVIDENCE
MAX_STALE_DAYS = 3    # page must be seen within this many days of the as-of date
TOP_N = 20

REPO_ID = "FlyRank/internship-warehouse"
LOCAL_DIR = "work/hf_data"
LOCAL_FACT = f"{LOCAL_DIR}/fact_content_daily_performance"

OUTPUT_DIR = "work/outputs"
OUTPUT_PATH = f"{OUTPUT_DIR}/baseline_action_score.csv"
CACHE_PATH = f"{OUTPUT_DIR}/cache_{START_DATE}_{END_DATE}.parquet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Token + preflight check ------------------------------------------
hf_token = os.environ.get("HF_TOKEN") or getpass("Enter your Hugging Face READ token: ")
hf_token = hf_token.strip()

who = requests.get("https://huggingface.co/api/whoami-v2",
                   headers={"Authorization": f"Bearer {hf_token}"})
if who.status_code != 200:
    raise PermissionError(
        "Hugging Face rejected the token (401). Create a new READ token, then run "
        "os.environ.pop('HF_TOKEN', None) and re-run this cell."
    )

# ---- Download only the needed months (skips files already present) ----
months = [str(p) for p in pd.period_range(START_DATE, END_DATE, freq="M")]
snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    token=hf_token,
    allow_patterns=[f"fact_content_daily_performance/month={m}/*.parquet" for m in months],
    local_dir=LOCAL_DIR,
)

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

print("Setup complete.")
print("Months:", months)
print("Historical data:", START_DATE, "->", END_DATE)
print("Prediction unit: next-day page search position")

Enter your Hugging Face READ token: ··········


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Setup complete.
Months: ['2026-06']
Historical data: 2026-06-01 -> 2026-06-30
Prediction unit: next-day page search position


# My rule and its reason codes

The application forecasts the next search location of every page.

I only use observations prior to the scoring date in this baseline.

No specific requirement for how the five months are accomplished. A page can make a contribution
The moment it has a good enough number of valid historical observations to make a directional
baseline.

Correct interpretation of search position:

- a lower position number corresponds to a better position number,
A larger position number will be a worse rank.

The baseline first attempts the next position based on recent observed
position behaviour.

The action score is then given more weight to pages that recently and
The predicted position means deteriorating.

Supporting signals such as impressions, CTR and observed activity are
The context is used instead of the target to be predicted.

It is a clear score for decision support, a baseline score.

It is NOT a probability, it is NOT the final trained ML prediction.

Reason codes:
- PREDICTED_RANK_DEGRADATION — next position is predicted as degradation.
- RECENT_RANK_DEGRADATION: recent observed position is deteriorating.
- LOW_CTR — CTR recently has been weak.
LOW_VISIBILITY — recent impressions are relatively low.
- INSUFFICIENT_EVIDENCE — Not enough observations useful for reaching a conclusion.
- NO_MAJOR_SIGNAL: No major negative signal found.

In [8]:
# Priority order matters: the first matching code becomes the primary reason.
REASON_CODES = [
    "INSUFFICIENT_EVIDENCE",
    "PERSISTENT_RANK_DETERIORATION",
    "PREDICTED_RANK_DEGRADATION",
    "LOW_CTR",
    "LOW_VISIBILITY",
    "NO_MAJOR_SIGNAL",
]

ACTION_LEVELS = ["HIGH_ATTENTION", "REVIEW", "MONITOR", "INSUFFICIENT_EVIDENCE"]

print("Reason codes:")
for c in REASON_CODES:
    print("-", c)
print("\nAction levels:")
for a in ACTION_LEVELS:
    print("-", a)

Reason codes:
- INSUFFICIENT_EVIDENCE
- PERSISTENT_RANK_DETERIORATION
- PREDICTED_RANK_DEGRADATION
- LOW_CTR
- LOW_VISIBILITY
- NO_MAJOR_SIGNAL

Action levels:
- HIGH_ATTENTION
- REVIEW
- MONITOR
- INSUFFICIENT_EVIDENCE


## 2. Build the ranked queue (writes the CSV)

The latest historical information is used to build the queue, page by page.

For every page:

1. Find its latest observed day.
2. Only make observations prior to the next prediction day.
3. Make an estimate of the next position based on past position behaviour.
4. Compare the estimated position to the most recent observed position.
5. Prioritize pages to be given greater importance of deteriorating ranks.
6. Provide CTR, impressions and activity as supporting evidence.

The length of the history available may vary from page to page.

If not exactly five, then at least five, pages in the navigation bar.
months of history.

This score is used to rank the pages from highest to lowest priority.
lowest priority.

In [9]:
KEYS = ["client_hash_id", "content_hash_id"]

# ---- Step 1: read (local files, cached) -------------------------------
print("Step 1/4: Reading daily page data...", flush=True)
t0 = time.time()

if os.path.exists(CACHE_PATH):
    df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded from cache: {len(df):,} rows ({time.time() - t0:.1f}s)", flush=True)
else:
    paths = ",".join(f"'{LOCAL_FACT}/month={m}/*.parquet'" for m in months)
    query = f"""
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        CASE
            WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
            THEN SUM(gsc_avg_position * COALESCE(gsc_impressions, 0))
                 / SUM(COALESCE(gsc_impressions, 0))
            ELSE AVG(gsc_avg_position)
        END AS avg_position,
        SUM(COALESCE(gsc_impressions, 0)) AS impressions,
        SUM(COALESCE(gsc_clicks, 0))      AS clicks
    FROM read_parquet([{paths}], union_by_name = true)
    WHERE gsc_avg_position IS NOT NULL
      AND gsc_avg_position > 0
      AND CAST(report_date AS DATE) BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'
    GROUP BY report_date, client_hash_id, content_hash_id
    """
    df = con.execute(query).df()
    df.to_parquet(CACHE_PATH, index=False)
    print(f"Data loaded: {len(df):,} rows ({time.time() - t0:.1f}s) -> cached", flush=True)

# ---- Step 2: prepare ---------------------------------------------------
print("Step 2/4: Preparing page observations...", flush=True)

df["report_date"] = pd.to_datetime(df["report_date"])
for c in ["avg_position", "impressions", "clicks"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["impressions"] = df["impressions"].fillna(0)
df["clicks"] = df["clicks"].fillna(0)

df = df[df["avg_position"].notna() & (df["avg_position"] > 0)].copy()
df = df.sort_values(KEYS + ["report_date"]).reset_index(drop=True)

AS_OF = df["report_date"].max()
assert AS_OF <= pd.Timestamp(END_DATE), "Data beyond END_DATE found"
print(f"Found {df['content_hash_id'].nunique():,} pages; as-of date {AS_OF.date()}", flush=True)

# ---- Step 3: features (vectorised, past data only) ---------------------
print("Step 3/4: Calculating page-level features...", flush=True)

g = df.groupby(KEYS, sort=False)
df["history_days"] = g["report_date"].transform("size")
df["pos_from_end"] = g.cumcount(ascending=False)
df["previous_position"] = g["avg_position"].shift(1)
df["delta"] = df["avg_position"] - df["previous_position"]

hist_mean = df.groupby(KEYS, sort=False)["avg_position"].mean()

rec = df[df["pos_from_end"] < RECENT_N].copy()
rec["x"] = rec.groupby(KEYS, sort=False).cumcount().astype(float)
rec["xy"] = rec["x"] * rec["avg_position"]
rec["xx"] = rec["x"] ** 2
rec["worse"] = (rec["delta"] > 0).astype(float).where(rec["delta"].notna())

agg = rec.groupby(KEYS, sort=False).agg(
    n=("x", "size"),
    sx=("x", "sum"),
    sy=("avg_position", "sum"),
    sxy=("xy", "sum"),
    sxx=("xx", "sum"),
    recent_position_mean=("avg_position", "mean"),
    recent_position_std=("avg_position", "std"),
    recent_impressions=("impressions", "mean"),
    recent_impr_sum=("impressions", "sum"),
    recent_clicks_sum=("clicks", "sum"),
    recent_worsening_share=("worse", "mean"),
)

# OLS slope over the recent window. Positive = position number rising = rank worse.
den = agg["n"] * agg["sxx"] - agg["sx"] ** 2
agg["position_slope"] = ((agg["n"] * agg["sxy"] - agg["sx"] * agg["sy"]) / den).where(agg["n"] >= 2)
agg["recent_ctr"] = (agg["recent_clicks_sum"] / agg["recent_impr_sum"]).where(agg["recent_impr_sum"] > 0)

last = df[df["pos_from_end"] == 0].set_index(KEYS)

queue_df = pd.DataFrame({
    "latest_date": last["report_date"],
    "prediction_date": last["report_date"] + pd.Timedelta(days=1),
    "history_days": last["history_days"],
    "latest_position": last["avg_position"],
    "previous_position": last["previous_position"],
    "historical_position_mean": hist_mean,
    "latest_impressions": last["impressions"],
    "latest_ctr": (last["clicks"] / last["impressions"]).where(last["impressions"] > 0),
    "recent_position_change": last["delta"] / last["previous_position"].abs(),
}).join(agg[[
    "recent_position_mean", "recent_position_std", "position_slope",
    "recent_impressions", "recent_ctr", "recent_worsening_share",
]])

# Baseline next-day prediction = latest + 0.4 * recent slope (damped trend). Position >= 1.
queue_df["baseline_predicted_position"] = (
    queue_df["latest_position"] + 0.4 * queue_df["position_slope"].fillna(0)
).clip(lower=1)
queue_df["predicted_position_change"] = (
    queue_df["baseline_predicted_position"] - queue_df["latest_position"]
)
queue_df["predicted_position_change_pct"] = (
    queue_df["predicted_position_change"] / queue_df["latest_position"].abs() * 100
)

queue_df = queue_df.reset_index()

# ---- Step 4: final queue ----------------------------------------------
print("Step 4/4: Creating final ranking queue...", flush=True)

queue_df["stale_days"] = (AS_OF - queue_df["latest_date"]).dt.days
n_before = len(queue_df)
queue_df = queue_df[queue_df["stale_days"] <= MAX_STALE_DAYS].copy()
print(f"Dropped {n_before - len(queue_df):,} stale pages (not seen within {MAX_STALE_DAYS} days of {AS_OF.date()})")

if queue_df.empty:
    raise ValueError("No eligible pages were found.")

queue_df = queue_df.sort_values("predicted_position_change", ascending=False).reset_index(drop=True)

print("\nCOMPLETE")
print("=" * 50)
print(f"Pages in queue : {len(queue_df):,}")
print(f"Rows in source : {len(df):,}")
print("=" * 50)
display(queue_df.head(10))

Step 1/4: Reading daily page data...
Data loaded: 3,806,731 rows (9.7s) -> cached
Step 2/4: Preparing page observations...
Found 201,853 pages; as-of date 2026-06-30
Step 3/4: Calculating page-level features...
Step 4/4: Creating final ranking queue...
Dropped 49,716 stale pages (not seen within 3 days of 2026-06-30)

COMPLETE
Pages in queue : 152,137
Rows in source : 3,806,731


,client_hash_id,content_hash_id,latest_date,prediction_date,history_days,latest_position,previous_position,historical_position_mean,latest_impressions,latest_ctr,...,recent_position_mean,recent_position_std,position_slope,recent_impressions,recent_ctr,recent_worsening_share,baseline_predicted_position,predicted_position_change,predicted_position_change_pct,stale_days
0,client_62f4a7e64f5e0096,content_0bda2cd98748d11c,2026-06-27,2026-06-28,2,290.0,8.0,149.000000,1.0,0.0,...,149.000000,199.404112,282.000000,1.0,0.0,1.0,402.800000,112.800000,38.896552,3
1,client_23a62021009f63c4,content_1e6f80df76add95d,2026-06-28,2026-06-29,2,296.0,19.0,157.500000,1.0,0.0,...,157.500000,195.868578,277.000000,2.0,0.0,1.0,406.800000,110.800000,37.432432,2
2,client_08a6a72ff48e62c0,content_f096936231ead73e,2026-06-30,2026-07-01,2,201.0,9.0,105.000000,1.0,0.0,...,105.000000,135.764502,192.000000,1.5,0.0,1.0,277.800000,76.800000,38.208955,0
3,client_3f0ce4d44fe94f3d,content_bf0ed236a2c4dc89,2026-06-29,2026-06-30,2,186.0,8.0,97.000000,1.0,0.0,...,97.000000,125.865007,178.000000,1.0,0.0,1.0,257.200000,71.200000,38.279570,1
4,client_62f4a7e64f5e0096,content_188e605214dccf09,2026-06-30,2026-07-01,3,257.0,7.0,89.666667,1.0,0.0,...,89.666667,144.918368,126.000000,1.0,0.0,1.0,307.400000,50.400000,19.610895,0
5,client_20259bd6705d81d4,content_c98cff5a7ea381a2,2026-06-29,2026-06-30,5,582.0,7.0,135.133333,1.0,0.0,...,135.133333,251.539438,109.566667,1.6,0.0,0.5,625.826667,43.826667,7.530355,1
6,client_62f4a7e64f5e0096,content_0939614ae1ebc3b6,2026-06-28,2026-06-29,2,99.0,1.0,50.000000,1.0,0.0,...,50.000000,69.296465,98.000000,1.0,0.0,1.0,138.200000,39.200000,39.595960,2
7,client_1a8bf67cad4ee525,content_5a34aef42a1de1fe,2026-06-27,2026-06-28,2,108.0,10.0,59.000000,1.0,0.0,...,59.000000,69.296465,98.000000,1.0,0.0,1.0,147.200000,39.200000,36.296296,3
8,client_73cda7b4e4f265ea,content_fff1dd9bc5d7fd9e,2026-06-29,2026-06-30,2,99.0,1.0,50.000000,1.0,0.0,...,50.000000,69.296465,98.000000,1.0,0.0,1.0,138.200000,39.200000,39.595960,1
9,client_e547b89c05043229,content_f73befc135ab2f50,2026-06-30,2026-07-01,2,94.0,2.0,48.000000,1.0,0.0,...,48.000000,65.053824,92.000000,1.0,0.0,1.0,130.800000,36.800000,39.148936,0


## 3. Top-20 review

The Top-20 list is a review queue, not a claim that these pages will
definitely lose rank.

For each page I show:

- current observed position,
- baseline next-position estimate,
- predicted position change,
- action level,
- reason code,
- confidence note,
- what could make the baseline wrong.

A higher priority means that the transparent baseline found stronger
evidence of possible rank deterioration.

The baseline does not provide a probability of ranking decline.

In [10]:
output_df = queue_df.copy()

def pct_of_positive(s):
    """Percentile among pages with a positive signal; non-positive -> 0."""
    s = s.fillna(0).clip(lower=0)
    return s.where(s > 0).rank(pct=True, method="average").fillna(0)

det_rank     = pct_of_positive(output_df["predicted_position_change"])
persist_rank = pct_of_positive(output_df["recent_worsening_share"])
recent_rank  = pct_of_positive(output_df["recent_position_change"])

# Directional decision-support score. NOT a probability.
output_df["action_score"] = 100 * (
    0.50 * det_rank + 0.25 * persist_rank + 0.25 * recent_rank
)

# ---- Action level ------------------------------------------------------
deteriorating = output_df["predicted_position_change"] > 0
enough = output_df["history_days"] >= MIN_HISTORY

output_df["action_level"] = "MONITOR"
output_df.loc[enough & deteriorating & (output_df["action_score"] >= 50), "action_level"] = "REVIEW"
output_df.loc[enough & deteriorating & (output_df["action_score"] >= 75), "action_level"] = "HIGH_ATTENTION"
output_df.loc[~enough, "action_level"] = "INSUFFICIENT_EVIDENCE"

# ---- Reason codes (all that apply, priority order; first = primary) ----
ctr_threshold = output_df["recent_ctr"].median()
vis_threshold = output_df["recent_impressions"].quantile(0.25)

flags = pd.DataFrame({
    "INSUFFICIENT_EVIDENCE": ~enough,
    "PERSISTENT_RANK_DETERIORATION": enough & deteriorating
        & (output_df["recent_position_change"] > 0)
        & (output_df["recent_worsening_share"] >= 0.5),
    "PREDICTED_RANK_DEGRADATION": enough & deteriorating,
    "LOW_CTR": enough & (output_df["recent_ctr"] < ctr_threshold),
    "LOW_VISIBILITY": enough & (output_df["recent_impressions"] < vis_threshold),
})[[c for c in REASON_CODES if c != "NO_MAJOR_SIGNAL"]]

output_df["all_reason_codes"] = flags.apply(
    lambda r: ";".join(r.index[r.to_numpy()]) or "NO_MAJOR_SIGNAL", axis=1
)
output_df["reason_code"] = output_df["all_reason_codes"].str.split(";").str[0]
assert set(output_df["reason_code"]).issubset(REASON_CODES)

# ---- Confidence / what could make it wrong ----------------------------
vol_threshold = output_df["recent_position_std"].quantile(0.90)
limited = output_df["history_days"] < 7
volatile = output_df["recent_position_std"] > vol_threshold

output_df["confidence_note"] = np.select(
    [limited, volatile],
    ["Limited history; baseline may be unstable.",
     "High recent position volatility; baseline may be unstable."],
    default="Directional baseline based on recent observed position history.",
)
output_df["could_be_wrong_if"] = np.select(
    [limited, volatile],
    ["Limited history may not represent normal page behaviour.",
     "Recent swings may be noise rather than trend."],
    default="Search position changes suddenly or recent history does not represent the next day.",
)

# ---- Leakage check (structural) ---------------------------------------
output_df["leakage_check_pass"] = (
    (output_df["latest_date"] <= pd.Timestamp(END_DATE))
    & (output_df["prediction_date"] > output_df["latest_date"])
)
assert output_df["leakage_check_pass"].all(), "Leakage check failed"

# ---- Final ranking + CSV ----------------------------------------------
output_df = (
    output_df
    .sort_values(["action_score", "predicted_position_change"], ascending=[False, False])
    .reset_index(drop=True)
)
output_df["priority_rank"] = np.arange(1, len(output_df) + 1)

output_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(output_df):,} rows -> {OUTPUT_PATH}")

top20 = output_df.head(TOP_N).copy()
display(top20[[
    "priority_rank", "content_hash_id", "latest_date", "latest_position",
    "baseline_predicted_position", "predicted_position_change",
    "predicted_position_change_pct", "action_score", "action_level",
    "reason_code", "confidence_note", "could_be_wrong_if",
]])
print("\nTop-20 pages available:", len(top20))

Saved 152,137 rows -> work/outputs/baseline_action_score.csv


,priority_rank,content_hash_id,latest_date,latest_position,baseline_predicted_position,predicted_position_change,predicted_position_change_pct,action_score,action_level,reason_code,confidence_note,could_be_wrong_if
0,1,content_0939614ae1ebc3b6,2026-06-28,99.000000,138.200000,39.200000,39.595960,99.843223,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
1,2,content_fff1dd9bc5d7fd9e,2026-06-29,99.000000,138.200000,39.200000,39.595960,99.843223,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
2,3,content_16147f2776bef728,2026-06-27,90.000000,125.600000,35.600000,39.555556,99.815928,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
3,4,content_f45816f297de3649,2026-06-27,67.000000,93.600000,26.600000,39.701493,99.810667,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
4,5,content_7bd06350f2468980,2026-06-29,74.000000,103.200000,29.200000,39.459459,99.752491,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
5,6,content_86a03b33991eede3,2026-06-29,58.333333,81.266667,22.933333,39.314286,99.683865,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
6,7,content_f73befc135ab2f50,2026-06-30,94.000000,130.800000,36.800000,39.148936,99.683274,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
7,8,content_7c3336fe470c3a2c,2026-06-28,93.500000,130.100000,36.600000,39.144385,99.678408,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
8,9,content_c93d965d2f7e4e5e,2026-06-27,56.500000,78.700000,22.200000,39.292035,99.670952,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...
9,10,content_e5c6dd3a9ead66ef,2026-06-27,54.000000,75.200000,21.200000,39.259259,99.654302,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,Limited history; baseline may be unstable.,Limited history may not represent normal page ...



Top-20 pages available: 20


## 4. Weak picks + leakage check
Some high-priority pages can be wrong.

Typical reasons include:

- very short history,
- high daily position volatility,
- an unusual one-day movement,
- missing measurements,
- a sudden external search change,
- a page-specific event that the baseline cannot observe.

The baseline should therefore be treated as a review mechanism, not as
ground truth.

The queue must use only information available before the prediction date.

The actual next-day position is not used to construct the baseline score.

No future target, intervention flag or product decision is used.

In [11]:
final_top20 = (
    output_df[output_df["leakage_check_pass"]]
    .sort_values(["action_score", "predicted_position_change"], ascending=[False, False])
    .head(TOP_N)
    .copy()
    .reset_index(drop=True)
)
final_top20["priority_rank"] = np.arange(1, len(final_top20) + 1)

def weak_flags(r):
    f = []
    if r["history_days"] < 7:
        f.append("SHORT_HISTORY")
    if pd.notna(r["recent_position_std"]) and r["recent_position_std"] > vol_threshold:
        f.append("HIGH_VOLATILITY")
    if pd.notna(r["recent_position_change"]) and abs(r["recent_position_change"]) > 0.5:
        f.append("ONE_DAY_JUMP")
    if pd.isna(r["recent_ctr"]):
        f.append("MISSING_CTR")
    return ";".join(f) or "NONE"

final_top20["weak_pick_flags"] = final_top20.apply(weak_flags, axis=1)

display(final_top20[[
    "priority_rank", "client_hash_id", "content_hash_id", "latest_date",
    "prediction_date", "latest_position", "baseline_predicted_position",
    "predicted_position_change", "predicted_position_change_pct",
    "position_slope", "action_score", "action_level", "reason_code",
    "history_days", "weak_pick_flags",
]])

print(f"\nFinal Top-20 queue created: {len(final_top20)} pages")
print("Pages with at least one weak-pick flag:", int((final_top20["weak_pick_flags"] != "NONE").sum()))

,priority_rank,client_hash_id,content_hash_id,latest_date,prediction_date,latest_position,baseline_predicted_position,predicted_position_change,predicted_position_change_pct,position_slope,action_score,action_level,reason_code,history_days,weak_pick_flags
0,1,client_62f4a7e64f5e0096,content_0939614ae1ebc3b6,2026-06-28,2026-06-29,99.000000,138.200000,39.200000,39.595960,98.000000,99.843223,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
1,2,client_73cda7b4e4f265ea,content_fff1dd9bc5d7fd9e,2026-06-29,2026-06-30,99.000000,138.200000,39.200000,39.595960,98.000000,99.843223,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
2,3,client_62f4a7e64f5e0096,content_16147f2776bef728,2026-06-27,2026-06-28,90.000000,125.600000,35.600000,39.555556,89.000000,99.815928,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
3,4,client_62f4a7e64f5e0096,content_f45816f297de3649,2026-06-27,2026-06-28,67.000000,93.600000,26.600000,39.701493,66.500000,99.810667,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
4,5,client_9958f0a7ae1df715,content_7bd06350f2468980,2026-06-29,2026-06-30,74.000000,103.200000,29.200000,39.459459,73.000000,99.752491,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
5,6,client_08a6a72ff48e62c0,content_86a03b33991eede3,2026-06-29,2026-06-30,58.333333,81.266667,22.933333,39.314286,57.333333,99.683865,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
6,7,client_e547b89c05043229,content_f73befc135ab2f50,2026-06-30,2026-07-01,94.000000,130.800000,36.800000,39.148936,92.000000,99.683274,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
7,8,client_62f4a7e64f5e0096,content_7c3336fe470c3a2c,2026-06-28,2026-06-29,93.500000,130.100000,36.600000,39.144385,91.500000,99.678408,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
8,9,client_2094c6eb080311d5,content_c93d965d2f7e4e5e,2026-06-27,2026-06-28,56.500000,78.700000,22.200000,39.292035,55.500000,99.670952,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP
9,10,client_62f4a7e64f5e0096,content_e5c6dd3a9ead66ef,2026-06-27,2026-06-28,54.000000,75.200000,21.200000,39.259259,53.000000,99.654302,INSUFFICIENT_EVIDENCE,INSUFFICIENT_EVIDENCE,2,SHORT_HISTORY;HIGH_VOLATILITY;ONE_DAY_JUMP



Final Top-20 queue created: 20 pages
Pages with at least one weak-pick flag: 20


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.